# Notebook 2 — ControlNet Training
Trains a ControlNet on mask→image pairs so the model learns to **follow a spleen mask shape**.

**Runtime:** ~4 hours on A100

## Cell 1 — Mount Drive and create folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/spleen_project"
os.makedirs(f"{BASE}/controlnet/data/images",              exist_ok=True)
os.makedirs(f"{BASE}/controlnet/data/conditioning_images", exist_ok=True)
os.makedirs(f"{BASE}/controlnet/output",                   exist_ok=True)
os.makedirs(f"{BASE}/controlnet/logs",                     exist_ok=True)

print("Folders ready.")

## Cell 2 — Install dependencies

In [ ]:
%%bash
pip install -q --upgrade diffusers transformers accelerate xformers datasets tensorboard safetensors Pillow
python -c "import diffusers; print('diffusers', diffusers.__version__)"
python -c "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())"
echo "Done"

## Cell 3 — Prepare image pairs
Both the ultrasound image and its matching mask are resized to 512×512 RGB.
Files keep their original names so pairs stay matched.

In [ ]:
from PIL import Image
import os

BASE     = "/content/drive/MyDrive/spleen_project"
IMG_SRC  = f"{BASE}/raw_images"
MASK_SRC = f"{BASE}/raw_masks"
IMG_OUT  = f"{BASE}/controlnet/data/images"
MASK_OUT = f"{BASE}/controlnet/data/conditioning_images"
SIZE     = 512

# Only process files that exist in BOTH folders
img_files  = sorted([f for f in os.listdir(IMG_SRC)  if f.endswith(".png")])
mask_files = sorted([f for f in os.listdir(MASK_SRC) if f.endswith(".png")])
matched    = sorted(set(img_files) & set(mask_files))
print(f"Matched pairs: {len(matched)}")

for i, fname in enumerate(matched):
    for src_dir, out_dir in [(IMG_SRC, IMG_OUT), (MASK_SRC, MASK_OUT)]:
        img = Image.open(os.path.join(src_dir, fname)).convert("L")
        img = img.resize((SIZE, SIZE), Image.LANCZOS)
        rgb = Image.merge("RGB", [img, img, img])
        rgb.save(os.path.join(out_dir, fname))  # keep original filename

    if i % 50 == 0:
        print(f"  {i+1}/{len(matched)}")

print(f"Done. {len(matched)} pairs written.")

## Cell 4 — Write metadata JSONL
The training script reads this file to know which image/mask pairs to load.
**Important:** use just the filename as the value, not the full path.

In [ ]:
import json, os

BASE      = "/content/drive/MyDrive/spleen_project"
IMG_OUT   = f"{BASE}/controlnet/data/images"
META_PATH = f"{BASE}/controlnet/data/train.jsonl"

img_files = sorted(os.listdir(IMG_OUT))

with open(META_PATH, "w") as f:
    for img in img_files:
        f.write(json.dumps({
            "image":              img,   # just filename, not full path
            "conditioning_image": img,   # mask has the same filename
            "text":               "ultrasound image"
        }) + "\n")

print(f"Written {len(img_files)} entries to train.jsonl")

# Verify first entry
with open(META_PATH) as f:
    print("First entry:", f.readline().strip())
# Expected: {"image": "ct1-1.png", "conditioning_image": "ct1-1.png", "text": "ultrasound image"}

## Cell 5 — Download training script + configure accelerate

In [ ]:
%%bash
cd /content
wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/controlnet/train_controlnet.py

cat > /content/accelerate_config.yaml << EOF
compute_environment: LOCAL_MACHINE
distributed_type: "NO"
gpu_ids: "0"
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
EOF

echo "Script ready: $(ls -lh /content/train_controlnet.py | awk '{print $5}')"
echo "Done"

## Cell 6 — Patch the training script
The downloaded script assumes images are pre-loaded PIL objects, but our JSONL contains file path strings.
This patch makes it open the file if it receives a string.

In [ ]:
with open("/content/train_controlnet.py", "r") as f:
    content = f.read()

# Patch: open image from path string if needed
old  = 'images = [image.convert("RGB") for image in examples[image_column]]'
new  = 'images = [Image.open(image).convert("RGB") if isinstance(image, str) else image.convert("RGB") for image in examples[image_column]]'

# Patch: open mask from path string if needed
old2 = 'conditioning_images = [image.convert("RGB") for image in examples[conditioning_image_column]]'
new2 = 'conditioning_images = [Image.open(image).convert("RGB") if isinstance(image, str) else image.convert("RGB") for image in examples[conditioning_image_column]]'

content = content.replace(old,  new)
content = content.replace(old2, new2)

with open("/content/train_controlnet.py", "w") as f:
    f.write(content)

print("✅ Patch applied" if new in content else "❌ Patch failed — string not found")

## Cell 7 — Run ControlNet training

**Before running:** paste keep-alive JS in browser console (F12):
```javascript
function ClickConnect() {
  console.log("Keeping alive...");
  document.querySelector("colab-toolbar-button#connect").click();
}
setInterval(ClickConnect, 60000);
```

**Key flags:**
- `--pretrained_model_name_or_path` — plain SD 1.5, no LoRA merge needed
- `--max_train_steps=15000` — ~4 hours on A100
- `--checkpointing_steps=1000` — saves to Drive every 1000 steps
- `--resume_from_checkpoint=latest` — just re-run this cell if disconnected

In [ ]:
%%bash
BASE="/content/drive/MyDrive/spleen_project"

accelerate launch \
  --config_file /content/accelerate_config.yaml \
  /content/train_controlnet.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --output_dir="$BASE/controlnet/output" \
  --train_data_dir="$BASE/controlnet/data" \
  --image_column="image" \
  --conditioning_image_column="conditioning_image" \
  --caption_column="text" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --learning_rate=1e-5 \
  --mixed_precision="fp16" \
  --max_train_steps=15000 \
  --checkpointing_steps=1000 \
  --validation_steps=1000 \
  --num_validation_images=2 \
  --validation_image="$BASE/controlnet/data/conditioning_images/ct1-1.png" \
  --validation_prompt="ultrasound image" \
  --enable_xformers_memory_efficient_attention \
  --resume_from_checkpoint="latest" \
  2>&1 | tee "$BASE/controlnet/logs/train_log.txt"